# PSR robustness to model parameter uncertainty (Table 4)

Monte Carlo perturbation of DH link lengths, masses, COM offsets; eps in {0%, 5%, 10%, 20%} plus geometry-only and inertial-only decompositions. Three seeds per non-zero level.


**Paper:** Zafar SA, Qin W. *Cross-task anomaly detection in reconfigurable industrial robot systems based on physics-structured regression of joint motor currents* (2026).

**Input dependency.** This notebook is a T4-row extension. It reads the existing 3-fold AUROC values from a CSV produced by an earlier run, computes the T4 row, and merges. Required input:

- `Processed_Data/NB8v6sl_robustness_detail.csv`

If the file is missing, the notebook raises an explicit error.


In [ ]:
import os, glob, time, warnings
import numpy as np
import pandas as pd
import h5py
import scipy.stats as sst
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")

ROOT = r"D:\Research\RCM"
BASE = os.path.join(ROOT, "Lab_Data")
OUT  = os.path.join(ROOT, "Processed_Data")

CANONICAL_DETAIL = os.path.join(OUT, "NB8v6sl_robustness_detail.csv")
T4_DETAIL_PATH   = os.path.join(OUT, "NB8v6sl_robustness_detail_T4.csv")
COMBINED_PATH    = os.path.join(OUT, "NB8v6sl_robustness_detail_4fold.csv")
SUMMARY_PATH     = os.path.join(OUT, "NB8v6sl_robustness_summary_4fold.csv")

# Constants
N_MC          = 3
N_DECOMP      = 3
PERT_LEVELS   = [0.00, 0.05, 0.10, 0.20]
PERT_DECOMP   = 0.10
TASKS_FULL    = ["T1", "T2", "T3", "T4"]
TASK_PAYLOAD  = {"T1": 0.0, "T2": 1.0, "T3": 3.0, "T4": 2.0}

PAYLOAD_COM  = np.array([0.0, 0.0, 0.05])
GRAVITY      = np.array([0.0, 0.0, -9.81])
RATE         = 125
SUBSAMPLE    = 4
MIN_SAMP     = 200

NOM_DH_A     = np.array([0, -0.42500, -0.39225, 0, 0, 0], dtype=float)
NOM_DH_D     = np.array([0.089159, 0.0, 0.0, 0.10915, 0.09465, 0.0823])
NOM_DH_ALPHA = np.array([np.pi/2, 0.0, 0.0, np.pi/2, -np.pi/2, 0.0])
NOM_MASS     = np.array([3.7000, 8.3930, 2.2750, 1.2190, 1.2190, 0.1879])
NOM_COM      = np.array([
    [0.0,     -0.02561,  0.00193],
    [0.21250,  0.0,      0.11336],
    [0.11993,  0.0,      0.02650],
    [0.0,     -0.00180,  0.01634],
    [0.0,      0.00180,  0.01634],
    [0.0,      0.0,     -0.00116],
])

FEAT_COLS = ([f"J{j}_{s}" for j in range(6)
              for s in ["resid_mean","resid_std","resid_rms","resid_max",
                        "resid_skew","resid_kurtosis",
                        "grav_resid_std","grav_resid_rms"]]
             + ["total_resid_rms","J1J2_resid_corr"])

# REGISTRY -- T1+T2+T3+T4 healthy & anomaly
REGISTRY = {
    "T1_healthy":    ("T1_PickPlace/Healthy",  "UR5_T1_healthy_180cyc_*.h5",          "T1","healthy","none",0.0),
    "T2_healthy":    ("T2_Assembly/Healthy",   "UR5_T2_healthy_180cyc_*.h5",          "T2","healthy","none",0.0),
    "T3_healthy":    ("T3_Palletize/Healthy",  "UR5_T3_healthy_183cyc_*.h5",          "T3","healthy","none",0.0),
    "T4_healthy":    ("T4_BinReorient/healthy","UR5_T4_healthy_session2_35cyc_*.h5",  "T4","healthy","none",0.0),
    "T1_A2_0.5kg":   ("T1_PickPlace/A2", "UR5_T1_A2_0.5kg_gripper_40cyc_*.h5",        "T1","A2","0.5kg",0.5),
    "T1_A2_1kg":     ("T1_PickPlace/A2", "UR5_T1_A2_1kg_gripper_40cyc_*.h5",          "T1","A2","1kg",1.0),
    "T1_A2_2kg":     ("T1_PickPlace/A2", "UR5_T1_A2_2kg_gripper_40cyc_*.h5",          "T1","A2","2kg",2.0),
    "T1_A3_10wraps": ("T1_PickPlace/A3", "UR5_T1_A3_1band_40cyc_*.h5",                "T1","A3","10wraps",0.0),
    "T1_A3_17wraps": ("T1_PickPlace/A3", "UR5_T1_A3_3bands_40cyc_*.h5",               "T1","A3","17wraps",0.0),
    "T1_A5_20mm":    ("T1_PickPlace/A5", "UR5_T1_A5_20mm_40cyc_*.h5",                 "T1","A5","20mm",0.0),
    "T1_A5_50mm":    ("T1_PickPlace/A5", "UR5_T1_A5_50mm_40cyc_*.h5",                 "T1","A5","50mm",0.0),
    "T1_A5_100mm":   ("T1_PickPlace/A5", "UR5_T1_A5_100mm_40cyc_*.h5",                "T1","A5","100mm",0.0),
    "T2_A2_1.5kg":   ("T2_Assembly/A2", "UR5_T2_A2_1.5kg_gripper_40cyc_*.h5",         "T2","A2","1.5kg",1.5),
    "T2_A2_2kg":     ("T2_Assembly/A2", "UR5_T2_A2_2kg_gripper_40cyc_*.h5",           "T2","A2","2kg",2.0),
    "T2_A2_3kg":     ("T2_Assembly/A2", "UR5_T2_A2_3kg_gripper_40cyc_*.h5",           "T2","A2","3kg",3.0),
    "T2_A3_7duct":   ("T2_Assembly/A3", "UR5_T2_A3_light_duct_40cyc_*.h5",            "T2","A3","7duct",0.0),
    "T2_A3_14duct":  ("T2_Assembly/A3", "UR5_T2_A3_medium_duct_40cyc_*_225508.h5",    "T2","A3","14duct",0.0),
    "T2_A5_20mm":    ("T2_Assembly/A5", "UR5_T2_A5_20mm_40cyc_*.h5",                  "T2","A5","20mm",0.0),
    "T2_A5_50mm":    ("T2_Assembly/A5", "UR5_T2_A5_50mm_40cyc_*.h5",                  "T2","A5","50mm",0.0),
    "T2_A5_100mm":   ("T2_Assembly/A5", "UR5_T2_A5_100mm_40cyc_*.h5",                 "T2","A5","100mm",0.0),
    "T3_A2_3.5kg":   ("T3_Palletize/A2", "UR5_T3_A2_3.5kg_gripper_33cyc_*.h5",        "T3","A2","3.5kg",3.5),
    "T3_A2_4kg":     ("T3_Palletize/A2", "UR5_T3_A2_4kg_gripper_33cyc_*.h5",          "T3","A2","4kg",4.0),
    "T3_A2_5kg":     ("T3_Palletize/A2", "UR5_T3_A2_4.5kg_gripper_33cyc_*.h5",        "T3","A2","5kg",5.0),
    "T3_A3_7duct":   ("T3_Palletize/A3", "UR5_T3_A3_light_duct_33cyc_*_222457.h5",    "T3","A3","7duct",0.0),
    "T3_A3_14duct":  ("T3_Palletize/A3", "UR5_T3_A3_medium_duct_33cyc_*_205648.h5",   "T3","A3","14duct",0.0),
    "T3_A5_20mm":    ("T3_Palletize/A5", "UR5_T3_A5_20mm_33cyc_*_172334.h5",          "T3","A5","20mm",0.0),
    "T3_A5_50mm":    ("T3_Palletize/A5", "UR5_T3_A5_50mm_33cyc_*_164447.h5",          "T3","A5","50mm",0.0),
    "T3_A5_100mm":   ("T3_Palletize/A5", "UR5_T3_A5_100mm_33cyc_*_160716.h5",         "T3","A5","100mm",0.0),
    "T4_A2_0.5kg":   ("T4_BinReorient/anomaly", "UR5_T4_A2_0.5kg_35cyc_*.h5",         "T4","A2","0.5kg",0.5),
    "T4_A2_1kg":     ("T4_BinReorient/anomaly", "UR5_T4_A2_1kg_35cyc_*.h5",           "T4","A2","1kg",1.0),
    "T4_A2_2kg":     ("T4_BinReorient/anomaly", "UR5_T4_A2_2kg_35cyc_*.h5",           "T4","A2","2kg",2.0),
    "T4_A3_7duct":   ("T4_BinReorient/anomaly", "UR5_T4_A3_7wraps_35cyc_*.h5",        "T4","A3","7duct",0.0),
    "T4_A3_14duct":  ("T4_BinReorient/anomaly", "UR5_T4_A3_14wraps_35cyc_*.h5",       "T4","A3","14duct",0.0),
    "T4_A5_20mm":    ("T4_BinReorient/anomaly", "UR5_T4_A5_20mm_35cyc_*.h5",          "T4","A5","20mm",0.0),
    "T4_A5_50mm":    ("T4_BinReorient/anomaly", "UR5_T4_A5_50mm_35cyc_*.h5",          "T4","A5","50mm",0.0),
    "T4_A5_100mm":   ("T4_BinReorient/anomaly", "UR5_T4_A5_100mm_35cyc_*.h5",         "T4","A5","100mm",0.0),
}

# Physics
def sample_perturbed_params(eps, rng, perturb_geometry=True,
                             perturb_inertial=True):
    """Sample one independent multiplicative perturbation per parameter.
    Ensures same seed produces the same parameter perturbations."""
    def perturb(arr):
        return arr * rng.uniform(1.0 - eps, 1.0 + eps, size=arr.shape)
    dh_a  = perturb(NOM_DH_A)  if perturb_geometry else NOM_DH_A.copy()
    dh_d  = perturb(NOM_DH_D)  if perturb_geometry else NOM_DH_D.copy()
    mass  = perturb(NOM_MASS)  if perturb_inertial else NOM_MASS.copy()
    com   = perturb(NOM_COM)   if perturb_inertial else NOM_COM.copy()
    alpha = NOM_DH_ALPHA.copy()
    return dh_a, dh_d, alpha, mass, com

def dh_transform(a, d, alpha, theta):
    ct, st = np.cos(theta), np.sin(theta)
    ca, sa = np.cos(alpha), np.sin(alpha)
    return np.array([
        [ct, -st*ca,  st*sa, a*ct],
        [st,  ct*ca, -ct*sa, a*st],
        [0,    sa,    ca,    d   ],
        [0,    0,     0,     1   ]
    ])

def gravity_torque_param(q, dh_a, dh_d, dh_alpha, mass, com, payload_mass=0.0):
    """Gravity torque via numerical Jacobian with explicit perturbed params."""
    T = [np.eye(4)]
    for i in range(6):
        T.append(T[-1] @ dh_transform(dh_a[i], dh_d[i], dh_alpha[i], q[i]))
    com_world = [(T[i+1] @ np.array([*com[i], 1.0]))[:3] for i in range(6)]
    masses = list(mass)
    if payload_mass > 0:
        masses.append(payload_mass)
        com_world.append((T[6] @ np.array([*PAYLOAD_COM, 1.0]))[:3])
    tau_g = np.zeros(6)
    dq = 1e-6
    for i in range(6):
        qp = q.copy(); qp[i] += dq
        Tp = [np.eye(4)]
        for jj in range(6):
            Tp.append(Tp[-1] @ dh_transform(dh_a[jj], dh_d[jj], dh_alpha[jj], qp[jj]))
        for jj in range(len(masses)):
            cp = ((Tp[jj+1] @ np.array([*com[jj], 1.0]))[:3] if jj < 6
                  else (Tp[6] @ np.array([*PAYLOAD_COM, 1.0]))[:3])
            tau_g[i] += masses[jj] * GRAVITY @ ((cp - com_world[jj]) / dq)
    return tau_g

def precompute_torques(cycles, dh_a, dh_d, dh_alpha, mass, com):
    for cyc in cycles:
        payload = TASK_PAYLOAD[cyc["task"]]
        q_a = cyc["q"]; qd_a = cyc["qd"]
        N = len(q_a)
        idx = list(range(0, N, SUBSAMPLE))
        n_sub = len(idx)
        tau_arr = np.zeros((n_sub, 6))
        qdd_arr = np.zeros((n_sub, 6))
        for ti, t in enumerate(idx):
            tau_arr[ti] = gravity_torque_param(q_a[t], dh_a, dh_d, dh_alpha,
                                                mass, com, payload_mass=payload)
            for j in range(6):
                qdd_arr[ti, j] = ((qd_a[t+1, j] - qd_a[t-1, j]) * RATE / 2.0
                                  if 0 < t < N - 1 else 0.0)
        cyc["tau_g"]   = tau_arr
        cyc["qdd"]     = qdd_arr
        cyc["sub_idx"] = idx

def clear_torque_cache(cycles):
    for cyc in cycles:
        cyc.pop("tau_g", None); cyc.pop("qdd", None); cyc.pop("sub_idx", None)

def fit_psr_fold(train_cycles):
    """Per-joint OLS (lstsq) on the M4 design matrix."""
    rows = {j: [] for j in range(6)}
    for cyc in train_cycles:
        qd_a = cyc["qd"]; cur = cyc["current"]
        tau_g_arr = cyc["tau_g"]; qdd_arr = cyc["qdd"]
        for ti, t in enumerate(cyc["sub_idx"]):
            for j in range(6):
                rows[j].append([tau_g_arr[ti, j], qd_a[t, j],
                                np.sign(qd_a[t, j]), qdd_arr[ti, j], 1.0,
                                cur[t, j]])
    psr_w = []
    for j in range(6):
        A = np.array(rows[j])
        w, _, _, _ = np.linalg.lstsq(A[:, :5], A[:, 5], rcond=None)
        psr_w.append(w)
    return psr_w

def extract_features_fold(cycles, psr_w):
    """50-dim PSR features (matches FEAT_COLS)."""
    feats = []
    for cyc in cycles:
        qd_a = cyc["qd"]; cur = cyc["current"]
        tau_g_arr = cyc["tau_g"]; qdd_arr = cyc["qdd"]
        n_sub = len(cyc["sub_idx"])
        res = np.zeros((n_sub, 6)); gr = np.zeros((n_sub, 6))
        for ti, t in enumerate(cyc["sub_idx"]):
            for j in range(6):
                phi = np.array([tau_g_arr[ti, j], qd_a[t, j],
                                np.sign(qd_a[t, j]), qdd_arr[ti, j], 1.0])
                res[ti, j] = cur[t, j] - phi @ psr_w[j]
                gr[ti, j]  = cur[t, j] - (psr_w[j][0]*tau_g_arr[ti, j]
                                          + psr_w[j][4])
        f = {}
        for j in range(6):
            r = res[:, j]; g = gr[:, j]
            f[f"J{j}_resid_mean"]     = r.mean()
            f[f"J{j}_resid_std"]      = r.std()
            f[f"J{j}_resid_rms"]      = np.sqrt(np.mean(r**2))
            f[f"J{j}_resid_max"]      = np.abs(r).max()
            f[f"J{j}_resid_skew"]     = float(sst.skew(r))
            f[f"J{j}_resid_kurtosis"] = float(sst.kurtosis(r))
            f[f"J{j}_grav_resid_std"] = g.std()
            f[f"J{j}_grav_resid_rms"] = np.sqrt(np.mean(g**2))
        f["total_resid_rms"] = np.sqrt(np.mean(res**2))
        f["J1J2_resid_corr"] = float(np.corrcoef(res[:, 1], res[:, 2])[0, 1]
                                      if len(res) > 2 else 0.0)
        f["is_anomaly"] = cyc["is_anomaly"]; f["anomaly"] = cyc["anomaly"]
        feats.append(f)
    return pd.DataFrame(feats)

def fold_rmse(train_cycles, psr_w):
    res2 = np.zeros(6); n_pts = 0
    for cyc in train_cycles:
        qd_a = cyc["qd"]; cur = cyc["current"]
        tau_g_arr = cyc["tau_g"]; qdd_arr = cyc["qdd"]
        for ti, t in enumerate(cyc["sub_idx"]):
            for j in range(6):
                phi = np.array([tau_g_arr[ti, j], qd_a[t, j],
                                np.sign(qd_a[t, j]), qdd_arr[ti, j], 1.0])
                pred = phi @ psr_w[j]
                res2[j] += (cur[t, j] - pred) ** 2
            n_pts += 1
    return np.sqrt(res2 / max(n_pts, 1))

# Detectors
def zscore_det(Xtr, Xte):
    mu = Xtr.mean(0); sg = Xtr.std(0) + 1e-8
    return np.abs((Xte - mu) / sg).mean(1)

def ocsvm_det(Xtr, Xte):
    sc = StandardScaler().fit(Xtr)
    clf = OneClassSVM(kernel="rbf", nu=0.05, gamma="scale")
    clf.fit(sc.transform(Xtr))
    return -clf.decision_function(sc.transform(Xte))

def if_det(Xtr, Xte):
    clf = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)
    clf.fit(Xtr)
    return -clf.decision_function(Xte)

def pca_det(Xtr, Xte):
    n = min(10, Xtr.shape[0]-1, Xtr.shape[1])
    sc = StandardScaler().fit(Xtr)
    pca = PCA(n_components=n).fit(sc.transform(Xtr))
    Xte_z = sc.transform(Xte)
    Xte_recon = pca.inverse_transform(pca.transform(Xte_z))
    return np.linalg.norm(Xte_z - Xte_recon, axis=1)

DETECTORS = {
    "Z-Score":    zscore_det,
    "OC-SVM":     ocsvm_det,
    "Iso-Forest": if_det,
    "PCA-Recon":  pca_det,
}

# T4-only LOTO function
def loto_t4(condition_label, tr_healthy, te_cycles):
    """Strict LOTO for T4 fold (train: T1+T2+T3 healthy already cached)."""
    records = []
    weights = fit_psr_fold(tr_healthy)
    df_tr = extract_features_fold(tr_healthy, weights)
    df_te = extract_features_fold(te_cycles, weights)
    Xtr = df_tr[FEAT_COLS].values
    Xte = df_te[FEAT_COLS].values
    y_te = df_te["is_anomaly"].values

    if y_te.sum() == 0 or y_te.sum() == len(y_te):
        return records, None

    # Per-anomaly Z-Score AUROC
    for anom in ["A2", "A3", "A5"]:
        mask_pa = np.array([c["is_anomaly"] == 0 or c["anomaly"] == anom
                            for c in te_cycles])
        y_pa = y_te[mask_pa]; Xte_pa = Xte[mask_pa]
        if y_pa.sum() == 0:
            continue
        sc_pa = zscore_det(Xtr, Xte_pa)
        try:
            auroc_pa = float(roc_auc_score(y_pa, sc_pa))
        except Exception:
            auroc_pa = float("nan")
        records.append({
            "detector":       "Z-Score",
            "fold_test_task": "T4",
            "anomaly_type":   anom,
            "auroc":          auroc_pa,
            "scope":          "per_anomaly",
            "condition":      condition_label,
        })

    # Aggregate AUROC for all detectors
    agg_zscore = None
    for dname, dfn in DETECTORS.items():
        try:
            sc = dfn(Xtr, Xte)
            auroc = float(roc_auc_score(y_te, sc))
        except Exception:
            auroc = float("nan")
        records.append({
            "detector":       dname,
            "fold_test_task": "T4",
            "anomaly_type":   "all",
            "auroc":          auroc,
            "scope":          "aggregate",
            "condition":      condition_label,
        })
        if dname == "Z-Score":
            agg_zscore = auroc

    # Training RMSE
    rmse = fold_rmse(tr_healthy, weights)
    rmse_record = {
        "detector":       "RMSE",
        "fold_test_task": "T4",
        "anomaly_type":   "healthy_rmse",
        "auroc":          float(rmse.mean()),
        "scope":          "rmse",
        "condition":      condition_label,
    }
    for j in range(6):
        rmse_record[f"rmse_J{j}"] = float(rmse[j])
    records.append(rmse_record)

    return records, agg_zscore

# Checkpoint helpers
def load_completed_t4():
    if not os.path.exists(T4_DETAIL_PATH):
        return set()
    df = pd.read_csv(T4_DETAIL_PATH)
    if df.empty:
        return set()
    return set(zip(df["eps"], df["condition"], df["seed"]))

def append_t4_records(records, eps, condition, seed):
    rows = []
    for r in records:
        row = {"eps": eps, "condition": condition, "seed": seed}
        row.update(r)
        rows.append(row)
    df_new = pd.DataFrame(rows)
    if os.path.exists(T4_DETAIL_PATH):
        df_new.to_csv(T4_DETAIL_PATH, mode="a", header=False, index=False)
    else:
        df_new.to_csv(T4_DETAIL_PATH, index=False)

# Main
print("=" * 70)
print("NB8v6_T4_extension -- robustness sweep for T4 fold")
print("=" * 70)

# Step 1: load 3-fold detail CSV (for the merge step at end)
print(f"\n[Step 1] Loading canonical {os.path.basename(CANONICAL_DETAIL)}...")
if not os.path.exists(CANONICAL_DETAIL):
    raise RuntimeError(
        f"Canonical detail CSV missing at {CANONICAL_DETAIL}. "
        "Cannot preserve T1/T2/T3 byte-for-byte without it."
    )
canon = pd.read_csv(CANONICAL_DETAIL)
print(f"  Canonical rows: {len(canon)}")
print(f"  Folds: {sorted(canon['fold_test_task'].unique())}")
print(f"  Conditions: {sorted(canon['condition'].unique())}")
print(f"  Eps levels: {sorted(canon['eps'].unique())}")

# Step 2: load all cycles
print("\n[Step 2] Loading HDF5 data (T1+T2+T3 healthy + T4 all)...")
all_cycles = []
for key, (subdir, pattern, task, anomaly, severity, _) in REGISTRY.items():
    matches = sorted(glob.glob(os.path.join(BASE, subdir, pattern)))
    if not matches:
        print(f"  WARNING -- not found: {key}")
        continue
    with h5py.File(matches[0], "r") as f:
        cnum    = f["cycle_number"][:].astype(int).ravel()
        cur_all = f["actual_current"][:]
        q_all   = f["actual_q"][:]
        qd_all  = f["actual_qd"][:]
    is_anom = 0 if anomaly == "healthy" else 1
    for c in np.unique(cnum[cnum > 0]):
        mask = cnum == c
        if mask.sum() >= MIN_SAMP:
            all_cycles.append({
                "q": q_all[mask], "qd": qd_all[mask],
                "current": cur_all[mask], "task": task,
                "anomaly": anomaly, "severity": severity,
                "is_anomaly": is_anom,
            })

# Filter: only need T1+T2+T3 healthy (for training) and T4 all (for testing)
tr_healthy = [c for c in all_cycles
              if c["task"] in ["T1","T2","T3"] and c["is_anomaly"] == 0]
te_cycles  = [c for c in all_cycles if c["task"] == "T4"]
print(f"  Training healthy (T1+T2+T3): {len(tr_healthy)} cycles")
print(f"  Test cycles (T4): {len(te_cycles)}")

# Step 3: run T4 robustness sweep
print("\n[Step 3] T4 robustness sweep...")
completed = load_completed_t4()
print(f"  Checkpoint: {len(completed)} T4 (eps, condition, seed) already done.")

# Main eps sweep
for eps in PERT_LEVELS:
    n_seeds = 1 if eps == 0.0 else N_MC
    for seed in range(n_seeds):
        condition = "all_params"
        if (eps, condition, seed) in completed:
            print(f"\n  [SKIP] eps={eps*100:.0f}%  seed={seed}  (done)")
            continue
        print(f"\n  --- eps={eps*100:.0f}%  seed={seed+1}/{n_seeds}  "
              f"condition={condition} ---")
        t0 = time.perf_counter()
        rng = np.random.default_rng(seed + 1000)  # SAME seed as canonical
        dh_a, dh_d, dh_alpha, mass, com = sample_perturbed_params(
            eps, rng, perturb_geometry=True, perturb_inertial=True)

        # Precompute torques for training + T4 cycles only
        precompute_torques(tr_healthy + te_cycles,
                            dh_a, dh_d, dh_alpha, mass, com)

        records, agg_z = loto_t4(condition, tr_healthy, te_cycles)
        append_t4_records(records, eps, condition, seed)
        clear_torque_cache(tr_healthy + te_cycles)

        elapsed = time.perf_counter() - t0
        if agg_z is not None:
            print(f"    T4 Z-Score AUROC = {agg_z:.4f}  ({elapsed:.1f}s)")
        print(f"  [CHECKPOINT] eps={eps*100:.0f}%  seed={seed}  saved.")

# Decomposition at eps=10%
print("\n" + "=" * 65)
print(f"Decomposition at eps={PERT_DECOMP*100:.0f}%: geometry_only vs inertial_only")
print("=" * 65)
for condition, geom, inert in [("geometry_only", True, False),
                                 ("inertial_only", False, True)]:
    print(f"\n  --- condition={condition} ---")
    for seed in range(N_DECOMP):
        if (PERT_DECOMP, condition, seed) in completed:
            print(f"    [SKIP] seed={seed}  (done)")
            continue
        print(f"    seed={seed+1}/{N_DECOMP}")
        t0 = time.perf_counter()
        rng = np.random.default_rng(seed + 2000)  # SAME seed as canonical
        dh_a, dh_d, dh_alpha, mass, com = sample_perturbed_params(
            PERT_DECOMP, rng, perturb_geometry=geom, perturb_inertial=inert)
        precompute_torques(tr_healthy + te_cycles,
                            dh_a, dh_d, dh_alpha, mass, com)
        records, agg_z = loto_t4(condition, tr_healthy, te_cycles)
        append_t4_records(records, PERT_DECOMP, condition, seed)
        clear_torque_cache(tr_healthy + te_cycles)
        elapsed = time.perf_counter() - t0
        if agg_z is not None:
            print(f"      T4 Z-Score AUROC = {agg_z:.4f}  ({elapsed:.1f}s)")
        print(f"    [CHECKPOINT] condition={condition}  seed={seed}  saved.")

# Step 4: merge 3-fold rows + T4 into combined detail CSV
print("\n[Step 4] Merging canonical (T1/T2/T3) + T4 into 4-fold detail CSV...")
t4_df = pd.read_csv(T4_DETAIL_PATH)
combined = pd.concat([canon, t4_df], ignore_index=True)
combined.to_csv(COMBINED_PATH, index=False)
print(f"  Saved: {COMBINED_PATH}  ({len(combined)} rows)")

# Integrity check
print("\n[Integrity check] T1/T2/T3 rows preserved byte-for-byte:")
canon_rows = combined[combined["fold_test_task"].isin(["T1","T2","T3"])]
match = len(canon_rows) == len(canon)
print(f"  {len(canon_rows)} T1/T2/T3 rows in combined (canonical had {len(canon)}). "
      f"{'OK' if match else 'MISMATCH'}")

# Step 5: build 4-fold summary table
print("\n[Step 5] Building 4-fold summary (Supp S6 format)...")
zs_agg = combined[(combined["detector"] == "Z-Score") &
                  (combined["scope"] == "aggregate")].copy()
print(f"  Z-Score aggregate rows: {len(zs_agg)}")

summary_rows = []
# Main sweep: all_params at each eps level
for eps in PERT_LEVELS:
    sub = zs_agg[(zs_agg["eps"] == eps) & (zs_agg["condition"] == "all_params")]
    if len(sub) == 0:
        continue
    # Mean across seeds for each fold
    fold_means = {}
    for fold in TASKS_FULL:
        vals = sub[sub["fold_test_task"] == fold]["auroc"].values
        fold_means[fold] = float(vals.mean()) if len(vals) > 0 else float("nan")
    overall_mean = float(np.nanmean(list(fold_means.values())))
    std_across_folds = float(np.nanstd(list(fold_means.values())))
    summary_rows.append({
        "Perturbation":  f"{int(eps*100)}%",
        "T1 AUROC":      round(fold_means["T1"], 4),
        "T2 AUROC":      round(fold_means["T2"], 4),
        "T3 AUROC":      round(fold_means["T3"], 4),
        "T4 AUROC":      round(fold_means["T4"], 4),
        "Mean AUROC":    round(overall_mean, 4),
        "Std":           round(std_across_folds, 4) if eps != 0.0 else "—",
    })

# Delta row: 0%→20%
if len(summary_rows) >= 4:
    r0  = summary_rows[0]
    r20 = summary_rows[3]
    summary_rows.append({
        "Perturbation":  "\u0394 (0%\u219220%)",
        "T1 AUROC":      f"{r20['T1 AUROC'] - r0['T1 AUROC']:+.4f}",
        "T2 AUROC":      f"{r20['T2 AUROC'] - r0['T2 AUROC']:+.4f}",
        "T3 AUROC":      f"{r20['T3 AUROC'] - r0['T3 AUROC']:+.4f}",
        "T4 AUROC":      f"{r20['T4 AUROC'] - r0['T4 AUROC']:+.4f}",
        "Mean AUROC":    f"{r20['Mean AUROC'] - r0['Mean AUROC']:+.4f}",
        "Std":           "—",
    })

# Decomposition rows
for cond_label, cond_key in [("\u03b5=10% (geometry only)", "geometry_only"),
                              ("\u03b5=10% (inertia only)",  "inertial_only")]:
    sub = zs_agg[(zs_agg["eps"] == PERT_DECOMP) & (zs_agg["condition"] == cond_key)]
    if len(sub) == 0:
        continue
    fold_means = {}
    for fold in TASKS_FULL:
        vals = sub[sub["fold_test_task"] == fold]["auroc"].values
        fold_means[fold] = float(vals.mean()) if len(vals) > 0 else float("nan")
    overall_mean = float(np.nanmean(list(fold_means.values())))
    std_across_folds = float(np.nanstd(list(fold_means.values())))
    summary_rows.append({
        "Perturbation":  cond_label,
        "T1 AUROC":      round(fold_means["T1"], 4),
        "T2 AUROC":      round(fold_means["T2"], 4),
        "T3 AUROC":      round(fold_means["T3"], 4),
        "T4 AUROC":      round(fold_means["T4"], 4),
        "Mean AUROC":    round(overall_mean, 4),
        "Std":           round(std_across_folds, 4),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_PATH, index=False)
print(f"  Saved: {SUMMARY_PATH}")
print()
print("=" * 95)
print("REVISED SUPP TABLE S6 (4-fold robustness)")
print("=" * 95)
print(summary_df.to_string(index=False))
print("=" * 95)

print("\nNB8v6_T4_extension complete.")